# Assignment 1: Custom Missing Value Imputer
**Course:** Feature Engineering & MLOps (Unit 1, Session 4)  
**Student Name:** Areeb Shah  
**Assignment No:** 1  
**Topic:** Object-Oriented Imputation Class following Scikit-Learn API  

---

### Objectives & Overview

In this assignment, I built and tested a custom scikit-learn transformer named `CustomImputer`. Building the transformer from scratch makes the imputation process and its safeguards explicit instead of hiding them behind a prebuilt utility.

Key requirements addressed in this notebook:
1. **Scikit-Learn API Standards:** Inherit from `BaseEstimator` and `TransformerMixin`, implement `fit()` and `transform()`, and ensure `fit()` returns `self` while `transform()` returns a modified copy.
2. **Train-Only Fitting Discipline:** Compute all fill statistics (median, mean, mode) strictly on the training partition to prevent data leakage.
3. **Automated Data Type Detection:** Automatically separate numeric features from categorical features using pandas data types without manual column lists.
4. **Missing Indicators for MNAR/MAR Data:** Add binary indicator columns (`<col>_was_missing`) to retain the signal that a value was originally absent.
5. **Guard Rails & Validation:** Ensure calling `transform()` before `fit()` raises a descriptive `NotFittedError`.
6. **Bonus (Per-Column Overrides):** Allow specific columns to use a different strategy than the dataset-wide default through a `column_overrides` dictionary.

## 1. Imports and Setup

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Display options and reproducible random seed
pd.set_option('display.max_columns', None)
np.random.seed(42)
print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Implementing the `CustomImputer` Class

Below is the implementation of `CustomImputer`. 
- In `fit()`, we inspect column data types with `np.issubdtype` and pandas dtypes. We calculate the fill value for each feature using the training data only and save them in `self.learned_fill_values_`.
- In `transform()`, we verify that the imputer has already been fitted using `check_is_fitted()`, add binary indicator flags for columns that had missing values in the training set, and replace nulls using our stored statistics.

In [2]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    Custom transformer that imputes missing values for mixed tabular datasets.
    
    Parameters:
    -----------
    numeric_strategy : str, default='median'
        Strategy for filling numeric columns ('median' or 'mean').
    categorical_strategy : str, default='most_frequent'
        Strategy for filling non-numeric columns ('most_frequent').
    add_missing_indicator : bool, default=True
        Whether to append binary indicator columns for features missing in train data.
    column_overrides : dict or None, default=None
        Optional dictionary mapping column names to specific strategies,
        e.g. {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}.
    """
    def __init__(self, numeric_strategy='median', categorical_strategy='most_frequent',
                 add_missing_indicator=True, column_overrides=None):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _determine_column_strategy(self, col_name, is_numeric):
        """Helper to resolve the effective strategy for a given column."""
        if self.column_overrides and col_name in self.column_overrides:
            return self.column_overrides[col_name]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    def fit(self, X, y=None):
        """
        Learn the fill values for each column from the training data X.
        """
        # Convert to DataFrame if not already
        df_in = pd.DataFrame(X).copy()
        
        self.feature_names_in_ = list(df_in.columns)
        
        # Auto-detect numeric vs categorical columns
        self.numeric_columns_ = [
            col for col in df_in.columns 
            if pd.api.types.is_numeric_dtype(df_in[col])
        ]
        self.categorical_columns_ = [
            col for col in df_in.columns 
            if col not in self.numeric_columns_
        ]
        
        # Track which columns in the training set actually had null values
        self.columns_with_missing_ = [
            col for col in df_in.columns if df_in[col].isnull().any()
        ]
        
        self.learned_fill_values_ = {}
        for col in df_in.columns:
            is_num = col in self.numeric_columns_
            strategy = self._determine_column_strategy(col, is_num)
            valid_vals = df_in[col].dropna()
            
            if is_num:
                if strategy == 'median':
                    self.learned_fill_values_[col] = valid_vals.median()
                elif strategy == 'mean':
                    self.learned_fill_values_[col] = valid_vals.mean()
                else:
                    raise ValueError(f"Unknown numeric strategy '{strategy}' for column '{col}'")
            else:
                if strategy == 'most_frequent':
                    modes = valid_vals.mode()
                    self.learned_fill_values_[col] = modes.iloc[0] if len(modes) > 0 else np.nan
                else:
                    raise ValueError(f"Unknown categorical strategy '{strategy}' for column '{col}'")
                    
        return self

    def transform(self, X):
        """
        Fill missing values using precomputed statistics and optionally add indicator columns.
        """
        # Guard rail: check if fitted before transforming
        check_is_fitted(self, 'learned_fill_values_')
        
        df_out = pd.DataFrame(X).copy()
        
        # Step 1: Add missingness indicators for columns that had missing values in train
        if self.add_missing_indicator:
            for col in self.columns_with_missing_:
                if col in df_out.columns:
                    df_out[f"{col}_was_missing"] = df_out[col].isnull().astype(int)
        
        # Step 2: Fill nulls with learned values (never recomputing statistics)
        for col in df_out.columns:
            if col in self.learned_fill_values_ and df_out[col].isnull().any():
                df_out[col] = df_out[col].fillna(self.learned_fill_values_[col])
                
        return df_out

## 3. Dataset Loading and Initial Missing Value Inspection

We read in the `student_performance_raw.csv` dataset and inspect which columns contain missing values.

In [3]:
# Support relative path whether executing from repo root or inside Assignments folder
data_file = '../data/raw/student_performance_raw.csv'
if not os.path.exists(data_file):
    data_file = 'data/raw/student_performance_raw.csv'

raw_df = pd.read_csv(data_file)
print(f"Dataset shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

# Summary of missing values
missing_counts = raw_df.isnull().sum()
cols_with_nulls = missing_counts[missing_counts > 0]
missing_overview = pd.DataFrame({
    'Null Count': cols_with_nulls,
    'Percentage': (cols_with_nulls / len(raw_df) * 100).round(2)
})
missing_overview

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/student_performance_raw.csv'

### Note on Missingness Mechanisms in this Dataset:
As discussed in the Unit 1 Session 4 lectures, this dataset has examples of all three missingness mechanisms:
- **MCAR (Missing Completely at Random):** `weekly_study_hours` is missing purely by chance, independent of study habits or performance.
- **MAR (Missing at Random):** `prev_exam_score` has missing values linked to student attendance (`attendance_pct`), an observed predictor.
- **MNAR (Missing Not at Random):** `mock_test_3` has missing values tied to its own underlying values (students who did poorly skipped reporting the test).
- **Categorical Missingness:** `income_bracket` and `feedback_text` also contain omitted answers.

## 4. Train / Test Split (Strict 80/20 Discipline)

We remove `student_id` since it is merely an identifier without predictive value, and separate `final_score` as our regression target. We perform the train/test split **before** any imputation to maintain clean separation between training and test distributions.

In [ ]:
predictor_cols = [c for c in raw_df.columns if c not in ('student_id', 'final_score')]

X = raw_df[predictor_cols].copy()
y = raw_df['final_score'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

## 5. Fitting on TRAIN Only and Transforming Both Sets

We fit our `CustomImputer` solely on `X_train`. The imputer records the computed statistics, which are then applied to both `X_train` and `X_test`.

In [ ]:
# Instantiate imputer with default settings
imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    add_missing_indicator=True
)

# Fit strictly on training split
imputer.fit(X_train)

# Transform both splits
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

print("Detected numeric columns:", imputer.numeric_columns_)
print("\nDetected categorical columns:", imputer.categorical_columns_)
print("\nColumns with missing values in train set:", imputer.columns_with_missing_)
print("\nLearned fill values:")
for col, val in imputer.learned_fill_values_.items():
    print(f"  {col}: {val}")

print(f"\nNew shape of X_train after indicators: {X_train_imputed.shape}")
print(f"New shape of X_test after indicators:  {X_test_imputed.shape}")

## 6. Verification Checks & Guard Rail Test

We verify that:
1. All null values have been successfully filled in the imputed feature columns across both train and test splits.
2. Attempting to call `transform()` before `fit()` correctly triggers a `NotFittedError`.

In [ ]:
# Check null counts in the original feature columns
train_nulls_left = X_train_imputed[imputer.feature_names_in_].isnull().sum().sum()
test_nulls_left = X_test_imputed[imputer.feature_names_in_].isnull().sum().sum()

print(f"Null values left in X_train: {train_nulls_left}")
print(f"Null values left in X_test:  {test_nulls_left}")

assert train_nulls_left == 0, "Assertion Error: Nulls still present in X_train!"
assert test_nulls_left == 0, "Assertion Error: Nulls still present in X_test!"
print("PASSED: 0 missing values remain in both train and test splits.")

In [ ]:
# Guard rail check: calling transform on an unfitted imputer
test_imputer = CustomImputer()
try:
    test_imputer.transform(X_train)
    print("FAILED: Guard rail did not raise error.")
except NotFittedError as err:
    print("PASSED: Guard rail caught unfitted transform as expected.")
    print("Error message:", err)

## 7. Distribution Impact: Mean & Standard Deviation Before vs. After Imputation

We look at how imputing missing values affects the mean and standard deviation of our numeric features.

In [ ]:
num_cols_missing = [c for c in imputer.numeric_columns_ if c in imputer.columns_with_missing_]

stats_rows = []
for col in num_cols_missing:
    raw_col = X_train[col]
    clean_col = X_train_imputed[col]
    stats_rows.append({
        'Feature': col,
        'Mean (Before)': raw_col.mean(),
        'Mean (After)': clean_col.mean(),
        'Mean Shift': clean_col.mean() - raw_col.mean(),
        'Std (Before)': raw_col.std(),
        'Std (After)': clean_col.std(),
        'Std Shift': clean_col.std() - raw_col.std(),
    })

stats_comparison_df = pd.DataFrame(stats_rows).round(4)
stats_comparison_df

### Commentary on Summary Statistics:
- **Central Tendency (Mean):** The mean shifts only very slightly across all features (e.g. `weekly_study_hours` shifts from 6.064 to 6.009). Because we used the median, the center of the distribution remains stable.
- **Dispersion (Standard Deviation):** The standard deviation consistently decreases for every column after imputation (e.g. `prev_exam_score` std drops from 14.37 to 14.02). This happens because replacing missing entries with a single constant value adds zero variance, pulling the overall spread of values closer to the center.

## 8. Sanity Check Against Scikit-Learn's `SimpleImputer`

To confirm that our custom implementation computes the exact correct fill values, we fit scikit-learn's built-in `SimpleImputer(strategy='median')` on the same training data and compare the results.

In [ ]:
sklearn_imputer = SimpleImputer(strategy='median')
sklearn_imputer.fit(X_train[num_cols_missing])

validation_table = pd.DataFrame({
    'Feature': num_cols_missing,
    'CustomImputer Value': [imputer.learned_fill_values_[c] for c in num_cols_missing],
    'SimpleImputer Value': sklearn_imputer.statistics_,
})
validation_table['Matches Exact'] = np.isclose(
    validation_table['CustomImputer Value'], validation_table['SimpleImputer Value']
)
print(validation_table)

assert validation_table['Matches Exact'].all(), "Sanity check failed: values do not match!"
print("\nPASSED: CustomImputer fill values match SimpleImputer exactly.")

## 9. Reflection Questions

**1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?**  
Calling `fit()` on the entire dataset or test set introduces data leakage by allowing information from unseen test data (such as the mean, median, or category frequencies) to influence how training values are processed. This makes evaluation metrics unrealistically optimistic and masks how the model will actually perform on fresh data. In real-world production setups, future samples do not exist yet when training, so fitting strictly on `X_train` ensures our pipeline reflects realistic deployment conditions.

**2. `mock_test_3` is MNAR (Session 4). Does mean/median imputation genuinely solve the problem for this column? What does your `add_missing_indicator` feature contribute that plain imputation does not?**  
No, median imputation does not genuinely solve the issue for an MNAR column like `mock_test_3`. In this dataset, students with poor scores or weak preparation were much more likely to skip or hide mock test 3, so filling their missing entries with the overall median (~71.3) artificially makes struggling students look like average performers. The `add_missing_indicator` column (`mock_test_3_was_missing`) helps by giving the model a separate binary feature indicating that the score was originally absent, allowing the algorithm to learn that the act of skipping the test is itself a strong signal of lower final performance.

**3. Suppose a brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do — and what SHOULD a production-grade version do instead?**  
In our current code, since the new column was never seen during `fit()`, it is not added to `self.learned_fill_values_`. When `transform()` runs, the check `if col in self.learned_fill_values_` evaluates to `False`, so the column passes through completely untouched with all of its missing values intact. In a production pipeline, this silent failure could break downstream models that expect zero nulls. A production-ready imputer should validate the input schema against expected columns and either raise an explicit schema validation error or follow a defined policy (such as dropping unexpected columns or imputing them with a documented fallback constant like 0 or 'missing').

## 10. Bonus (+10% Extra Credit): Per-Column Strategy Overrides

We showcase the `column_overrides` feature by assigning a different strategy to `weekly_study_hours` (`mean`) while keeping the dataset default (`median`) for the other numeric features.

In [ ]:
override_imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    column_overrides={
        'weekly_study_hours': 'mean',       # Overridden to mean
        'income_bracket': 'most_frequent'  # Explicit categorical override
    }
)

override_imputer.fit(X_train)

default_study_val = imputer.learned_fill_values_['weekly_study_hours']
override_study_val = override_imputer.learned_fill_values_['weekly_study_hours']

print(f"Default numeric fill value for weekly_study_hours (median): {default_study_val:.4f}")
print(f"Overridden fill value for weekly_study_hours (mean):        {override_study_val:.4f}")
print(f"income_bracket fill value:                                 {override_imputer.learned_fill_values_['income_bracket']}")

assert default_study_val != override_study_val, "Override failed to change fill value!"
print("\nPASSED: Column override correctly modified weekly_study_hours strategy without altering other columns.")